**qaoa_optimizer_test**
This notebook checks if the QAOA algorithm can find better solutions as we make the circuit deeper by increasing the number of layers ($p$). It also analyzes how hardware noise disrupts the results and compares different math methods to find the most efficient way to tune the algorithm's settings.


In [1]:
!pip install qiskit_aer


In [12]:
#imports
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from solver.quantum_solver.qaoa_solver.qaoa_optimizer import QAOALocalOptimizer

In [13]:
#Setup problem and simulator
s1, s2, s3, s4, s5, s6 = sp.symbols('s1 s2 s3 s4 s5 s6')
n_qubits = 6
problem = s1*s2 + s2*s3 + s3*s4 + s4*s5 + s5*s6 # Small test instance 
sim_ideal = AerSimulator()

#Test different depths (p) [cite: 117]
p_values = [1, 2, 3, 4]
results = []

for p in p_values:
    optimizer = QAOALocalOptimizer(sim_ideal, (0, np.pi), (0, np.pi), p, 1024, 'COBYLA')
    best_cost, _ = optimizer.optimize(problem, p) 
    results.append(best_cost)

AttributeError: 'Add' object has no attribute 'variables'

In [ ]:
#Adding Noise
noise_model = NoiseModel()
error = depolarizing_error(0.01, 1) # 1% error
noise_model.add_all_qubit_quantum_error(error, ['rx', 'rz', 'sx'])
sim_noisy = AerSimulator(noise_model=noise_model)

optimizer_noisy = QAOALocalOptimizer(sim_noisy, (0, np.pi), (0, np.pi), 1, 1024, 'COBYLA')
noisy_cost, _ = optimizer_noisy.optimize(problem, 1)

In [ ]:
#Visualization
plt.plot(p_values, results, marker='o', label='Ideal')
plt.axhline(y=noisy_cost, color='r', linestyle='--', label='Noisy (p=1)')
plt.xlabel('Depth (p)')
plt.ylabel('Expectation Value')
plt.legend()
plt.show()